In [1]:
import event_helpers
import gcsfs
import geopandas as gpd
import google.auth
import pandas as pd
import poi_geography_utils as poi_geog_utils
import world_cup_vars as wc_vars
from gtfs_curator_utils import utils

GCS_FILE_PATH = wc_vars.GCS_FILE_PATH
credentials, _ = google.auth.default()

In [2]:
STADIUM_NAME = "sofi"
route_gdf = gpd.read_parquet(
    f"{GCS_FILE_PATH}route_summary_{STADIUM_NAME}.parquet",
    storage_options = {"token": credentials.token}
)

In [3]:
route_gdf.head()

,schedule_name,route_name,direction_id,route_type,shape_array_key,geometry,daily_trips_weekday_non_event,daily_trips_weekend_non_event,daily_trips_weekday_event,daily_trips_weekend_event,...,change_tu_messages_per_minute_weekday,change_tu_messages_per_minute_weekend,combined_change_daily_trips,combined_change_pct_vp_trips,combined_change_vp_messages_per_minute,combined_change_pct_tu_trips,combined_change_tu_messages_per_minute,is_route_near,point_of_interest,is_special_route
0,Beach Cities GMV Schedule,4815__109,0,3,374f0a8ed36e1c4af51a3c1cb98d5cec,"LINESTRING (-118.38527 33.81654, -118.38554 33...",21.00,16.0,21.0,16.0,...,-0.0,-0.0,0.0,0.0,0.0,0.1,0.0,True,SoFi Stadium,False
1,Beach Cities GMV Schedule,4815__109,1,3,73ee3ae44a9abd2cbd5ddafad8fc299b,"LINESTRING (-118.37778 33.95230, -118.37765 33...",21.00,16.0,21.0,16.0,...,-0.0,-0.1,0.0,-0.1,-0.2,0.1,-0.1,True,SoFi Stadium,False
2,Beach Cities GMV Schedule,4819__102,0,3,3b4c863d19c339d4cc6a6419100e6f8c,"LINESTRING (-118.39031 33.83858, -118.39024 33...",27.00,22.0,27.0,17.0,...,-0.1,0.2,-5.0,-0.1,0.1,0.0,0.1,False,None,False
3,Beach Cities GMV Schedule,4819__102,1,3,8e5aa4fb2d7026a6a53a30ab1a4cc925,"LINESTRING (-118.36983 33.89507, -118.36992 33...",34.00,16.0,34.0,11.0,...,-0.1,0.1,-5.0,0.0,0.0,0.0,0.0,False,None,False
4,Big Blue Bus Schedule,14__14 Bundy Drive & Centinela Avenue,0,3,3a784bf21725466ea6e9a2c9b6a97e48,"LINESTRING (-118.46385 34.05627, -118.46398 34...",59.27,33.0,59.0,33.0,...,0.0,0.0,-0.3,0.0,0.0,0.0,0.0,False,None,False


## RT completeness, 
* maybe just show `pct_vp/tu_trips` and `vp/tu_messages_per_minute` for event period?
* not important to show non-event or change

In [4]:
import gt_extras as gte
from great_tables import GT, md

In [5]:
def define_rt_columns(rt_type: str):
    return [f"pct_{rt_type}_trips_weekday_event", f"pct_{rt_type}_trips_weekend_event",
      f"{rt_type}_messages_per_minute_weekday_event", f"{rt_type}_messages_per_minute_weekend_event"
    ] 

In [6]:
route_cols = ["schedule_name", "route_name"]
vp_cols = define_rt_columns("vp")
tu_cols = define_rt_columns("tu")
vp_cols

['pct_vp_trips_weekday_event',
 'pct_vp_trips_weekend_event',
 'vp_messages_per_minute_weekday_event',
 'vp_messages_per_minute_weekend_event']

In [13]:
rt_table = (
    GT(route_gdf[route_gdf.is_special_route==True][route_cols + vp_cols + tu_cols])
    .cols_label(
        tu_messages_per_minute_weekday_event = "Trip Updates per Minute",
        pct_tu_trips_weekday_event = "% Trips Weekday",
        pct_tu_trips_weekend_event = "% Trips Weekend",
        vp_messages_per_minute_weekday_event = "Vehicle Positions per Minute",
        pct_vp_trips_weekday_event = "% Trips Weekday",
        pct_vp_trips_weekend_event = "% Trips Weekend",
    ).fmt_number(
        columns = ["tu_messages_per_minute_weekday_event", "vp_messages_per_minute_weekday_event"], 
        decimals=1
    ).fmt_percent(
        columns=["pct_tu_trips_weekday_event", "pct_tu_trips_weekend_event", 
                 "pct_vp_trips_weekday_event", "pct_vp_trips_weekend_event"], 
        decimals=1
    ).tab_spanner(
        label="Trip Updates",
        columns=tu_cols
    ).tab_spanner(
        label="Vehicle Positions",
        columns=vp_cols
    )
)

rt_table.pipe(
    gte.gt_color_box, 
    columns=["tu_messages_per_minute_weekday_event", "vp_messages_per_minute_weekday_event"], 
    palette="Blues",
    domain=[1, 4]
).pipe(
    gte.gt_hulk_col_numeric, 
    columns=["pct_tu_trips_weekday_event", "pct_tu_trips_weekend_event", 
             "pct_vp_trips_weekday_event", "pct_vp_trips_weekend_event"],
    palette=["#FFEC8B", "#E5F5E0"], #[light goldenrod1, white alyssum (from greens)]
    domain=[0, 1],
    alpha=0.1
)

/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/gt_extras/_utils_column.py:127: UserWarning: Value 0.94 in column 'tu_messages_per_minute_weekday_event' is less than the domain minimum 1. Setting to 1.
  warnings.warn(
/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/gt_extras/_utils_column.py:127: UserWarning: Value 0.0 in column 'tu_messages_per_minute_weekday_event' is less than the domain minimum 1. Setting to 1.
  warnings.warn(
/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/gt_extras/_utils_column.py:127: UserWarning: Value 0.0 in column 'vp_messages_per_minute_weekday_event' is less than the domain minimum 1. Setting to 1.
  warnings.warn(


GT(_tbl_data=                schedule_name  \
37      Big Blue Bus Schedule   
38      Big Blue Bus Schedule   
74           G Trans Schedule   
75           G Trans Schedule   
148           LA DOT Schedule   
149           LA DOT Schedule   
395  LA Metro Events Schedule   
396  LA Metro Events Schedule   
397  LA Metro Events Schedule   
398  LA Metro Events Schedule   
399  LA Metro Events Schedule   
400  LA Metro Events Schedule   
401  LA Metro Events Schedule   
402  LA Metro Events Schedule   
403  LA Metro Events Schedule   
404  LA Metro Events Schedule   
405  LA Metro Events Schedule   
406  LA Metro Events Schedule   
407  LA Metro Events Schedule   
408  LA Metro Events Schedule   
409  LA Metro Events Schedule   
410  LA Metro Events Schedule   
411  LA Metro Events Schedule   
412  LA Metro Events Schedule   
413  LA Metro Events Schedule   
414  LA Metro Events Schedule   
415  LA Metro Events Schedule   
416  LA Metro Events Schedule   
417  LA Metro Events Schedule   
418  LA Metro Events Schedule   
419  LA Metro Events Schedule   
420  LA Metro Events Schedule   
421  LA Metro Events Schedule   
422  LA Metro Events Schedule   
423  LA Metro Events Schedule   
424  LA Metro Events Schedule   
429    LA Metro Rail Schedule   
430    LA Metro Rail Schedule   
435    LA Metro Rail Schedule   
436    LA Metro Rail Schedule   
437        Metrolink Schedule   
438        Metrolink Schedule   
439        Metrolink Schedule   
440        Metrolink Schedule   
443        Metrolink Schedule   
444        Metrolink Schedule   
445        Metrolink Schedule   
446        Metrolink Schedule   
447        Metrolink Schedule   
448        Metrolink Schedule   
449        Metrolink Schedule   
450        Metrolink Schedule   
451         Torrance Schedule   
452         Torrance Schedule   

                                            route_name  \
37                        T14__T14 Los Angeles Stadium   
38                        T14__T14 Los Angeles Stadium   
74                                      7X__7X Line 7X   
75                                      7X__7X Line 7X   
148                      712__DASH Chesterfield Square   
149                      712__DASH Chesterfield Square   
395             10__ S10 Harbor Gateway Transit Center   
396             10__ S10 Harbor Gateway Transit Center   
397                  11__ S11 LAX/Metro Transit Center   
398                  11__ S11 LAX/Metro Transit Center   
399                      12__ S12 Hotels & Parking LAX   
400                      12__ S12 Hotels & Parking LAX   
401                    13__ T13 Pierce College Station   
402                    13__ T13 Pierce College Station   
403                     14__ T14 Downtown Santa Monica   
404                     14__ T14 Downtown Santa Monica   
405                   15__ T15 North Hollywood Station   
406                   15__ T15 North Hollywood Station   
407                           1__ R1 El Camino College   
408                           1__ R1 El Camino College   
409                               2__ R2 Union Station   
410                               2__ R2 Union Station   
411                            3__ R3 Crenshaw Station   
412                            3__ R3 Crenshaw Station   
413                    4__ R4 Hawthorne/Lennox Station   
414                    4__ R4 Hawthorne/Lennox Station   
415                         5__ R5 Downtown Long Beach   
416                         5__ R5 Downtown Long Beach   
417                       6__ R6 ARTIC Anaheim Station   
418                       6__ R6 ARTIC Anaheim Station   
419               7__ R7 Newport Transportation Center   
420               7__ R7 Newport Transportation Center   
421                     8__ S8 Torrance Transit Center   
422                     8__ S8 Torrance Transit Center   
423                  9__ S9 Culver City Transit Center   
424                  9__ S9 Culver City Transit Center   
429                      

In [ ]:
trips_wide_gdf_near[trips_wide_gdf_near.is_special_route==False].combined_change_pct_vp_trips.hist()

In [ ]:
trips_wide_gdf_near[trips_wide_gdf_near.is_special_route==True].combined_change_pct_vp_trips.hist()